**Programmer:** python_scripts (Abhijith Warrier)

**PYTHON SCRIPT TO _TRANSLATE TEXT, SUMMARIZE IT WITH A PRETRAINED MODEL, AND GENERATE HASHTAGS FROM THE SUMMARY._** 🐍🌐📝#️⃣

This notebook chains three mini AI tasks:
1) **Translate** any language → English (no API key)
2) **Summarize** the English text (Hugging Face Transformers)
3) **Generate Hashtags** (keyword extraction with TF-IDF)
Optionally, it can translate the **summary + hashtags back** to your target language.

### 📦 Install Dependencies

In [1]:
# Install lightweight deps (uncomment if needed)
# !pip install -q deep-translator transformers scikit-learn

### 🧰 Imports

In [ ]:
from deep_translator import GoogleTranslator
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import string

### 📝 Input Text & Languages
- `src_lang`: the language of your original text (e.g., "auto" to auto-detect)
- `pivot_lang`: language to summarize in (usually English: "en")
- `tgt_lang`: optional language to translate the final outputs back to

In [3]:
# Example multilingual input (replace with your own)
original_text = """
La inteligencia artificial está transformando industrias enteras, desde la salud hasta las finanzas.
Permite automatizar tareas, encontrar patrones complejos y ofrecer experiencias personalizadas a gran escala.
"""

src_lang = "auto"  # detect automatically
pivot_lang = "en"  # summarize in English
tgt_lang = "en"  # change (e.g., "es", "fr", "de") to translate final outputs back

### 🌐 Step 1 — Translate to English (Pivot)
We translate the original text to the **pivot language** (default English) for better summarization quality.

In [4]:
def translate_text(text: str, source: str, target: str) -> str:
    """
    Translate `text` from `source` → `target` using GoogleTranslator (no API key).
    Set source="auto" to auto-detect the input language.
    """
    return GoogleTranslator(source=source, target=target).translate(text)


translated_en = translate_text(original_text, src_lang, pivot_lang)
print("=== Translated → English (preview) ===\n", translated_en[:400], "...\n")

=== Translated → English (preview) ===
 Artificial intelligence is transforming entire industries, from health to finance.
It allows automating tasks, finding complex patterns and offering personalized experiences on a large scale. ...



### ✂️ Step 2 — Summarize (Hugging Face)
Use a compact summarization model for speed and quality.
- Default: `sshleifer/distilbart-cnn-12-6` (fast BART-based summarizer)

In [5]:
# Build the summarization pipeline once (downloads model on first run)
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")


def summarize(text: str, max_chars: int = 1800) -> str:
    """
    Summarize text. If it's very long, we truncate for demo purposes.
    For production, chunk the text and summarize per chunk, then summarize the summaries.
    """
    text = text.strip()
    if len(text) > max_chars:
        text = text[:max_chars]
    out = summarizer(text, max_length=130, min_length=45, do_sample=False)
    return out[0]["summary_text"].strip()


summary_en = summarize(translated_en)
print("=== Summary (EN) ===\n", summary_en, "\n")

Device set to use mps:0
Your max_length is set to 130, but your input_length is only 33. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=16)


=== Summary (EN) ===
 Artificial intelligence is transforming entire industries, from health to finance . It allows automating tasks, finding complex patterns and offering personalized experiences on a large scale . Artificial intelligence allows automation of tasks and finding patterns in complex patterns . 



### #️⃣ Step 3 — Generate Hashtags from the Summary
We extract salient keywords with **TF-IDF**, then turn them into simple, readable hashtags.

In [6]:
def simple_clean(text: str) -> str:
    """Lowercase, remove punctuation/numbers, and collapse spaces."""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)  # remove URLs
    text = text.translate(str.maketrans("", "", string.punctuation + string.digits))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def top_keywords_tfidf(text: str, k: int = 8):
    """
    Return top-k keywords using TF-IDF on a single document by splitting into sentences.
    (Crude but effective for short social captions.)
    """
    # Split into pseudo-documents (sentences) to let TF-IDF score terms
    sentences = re.split(r"[.!?]\s+", text)
    sentences = [s for s in sentences if s.strip()]
    if not sentences:
        sentences = [text]

    vec = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),  # allow unigrams + bigrams
        max_features=1000,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",  # alphabetic tokens (≥2 letters)
    )
    X = vec.fit_transform(sentences)
    # Aggregate scores across sentences
    scores = X.sum(axis=0).A1
    terms = vec.get_feature_names_out()
    ranked = sorted(zip(terms, scores), key=lambda x: x[1], reverse=True)[:k]
    return [w for w, _ in ranked]


def to_hashtags(words):
    """Convert keyword tokens/phrases to social-friendly #hashtags."""
    tags = []
    for w in words:
        token = re.sub(r"\s+", "", w)  # remove spaces for bigrams
        token = re.sub(r"[^a-zA-Z0-9]", "", token)
        if token:
            tags.append("#" + token[:28])  # keep tags readable
    # De-duplicate while preserving order
    seen = set()
    uniq = []
    for t in tags:
        if t.lower() not in seen:
            uniq.append(t)
            seen.add(t.lower())
    return uniq


keywords = top_keywords_tfidf(simple_clean(summary_en), k=10)
hashtags_en = to_hashtags(keywords)
print("=== Hashtags (EN) ===\n", " ".join(hashtags_en), "\n")

=== Hashtags (EN) ===
 #patterns #allows #artificial #artificialintelligence #complex #complexpatterns #finding #intelligence #tasks #tasksfinding 



### 🚀 End-to-End Demo (One Function)
Pass any text + languages to get translated summary + hashtags in one go.

In [7]:
def translate_summarize_hashtag(text: str, src="auto", pivot="en", tgt="en", k=10):
    # 1) translate → English
    en = translate_text(text, src, pivot)
    # 2) summarize (EN)
    summ_en = summarize(en)
    # 3) hashtags from EN summary
    kws = top_keywords_tfidf(simple_clean(summ_en), k=k)
    tags_en = to_hashtags(kws)
    # 4) optional translate back
    out_summary = summ_en if tgt == "en" else translate_text(summ_en, "en", tgt)
    _, out_tags = (tags_en, tags_en) if tgt == "en" else maybe_translate_list(tags_en, "en", tgt)
    return out_summary, out_tags


demo_summary, demo_tags = translate_summarize_hashtag(original_text, src=src_lang, pivot=pivot_lang, tgt=tgt_lang, k=10)
print("=== DEMO SUMMARY ===\n", demo_summary, "\n")
print("=== DEMO HASHTAGS ===\n", " ".join(demo_tags))

Your max_length is set to 130, but your input_length is only 33. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=16)


=== DEMO SUMMARY ===
 Artificial intelligence is transforming entire industries, from health to finance . It allows automating tasks, finding complex patterns and offering personalized experiences on a large scale . Artificial intelligence allows automation of tasks and finding patterns in complex patterns . 

=== DEMO HASHTAGS ===
 #patterns #allows #artificial #artificialintelligence #complex #complexpatterns #finding #intelligence #tasks #tasksfinding


### 📝 Notes & Tips
- **Translation**: `deep-translator` uses public endpoints; great for demos (no key).
- **Summarization**: The first run downloads the model; for longer texts, process in **chunks**.
- **Hashtags**: TF-IDF on the summary keeps tags **concise & relevant**. Adjust `ngram_range`/`k`.
- **Localization**: Set `tgt_lang` to your audience’s language (e.g., `"es"`, `"fr"`, `"hi"`) to publish localized posts.
- **Alternatives**: For keywording, you can try **KeyBERT** or **YAKE**; for faster summarization, try **`facebook/bart-base`** or **`t5-small`**.